
Copyright (c) Meta Platforms, Inc. and affiliates.
All rights reserved.

This source code is licensed under the license found in the
LICENSE file in the root directory of this source tree.

# LT-Swap Benchmarking - LLM Inference Notebook

#### Purpose

This notebook has been created to facilitate the LLM-depent part of the LT-Swap benchmarking pipeline (for generation and filtering).

Prompting LLM APIs, can become challenging due to rate limiting, API errors, network issues and others. This notebook leverages batching, a queue, asynchronous workers and retries to maximize the robustness of this step of the LT-Swap benchmarking pipeline.

#### Making this notebook work with your LLM API

This notebook relies on two scripts 'mp_main.py' and 'mp_utils.py'. To make it compatible with your LLM API, you need to create your own subclass of *LLMClient* in 'mp_main.py' and also customize the function main() to setup the new client.

Illustrative subclasses have been provided for OpenAI API, Azure OpenAI API and [Matrix](https://github.com/facebookresearch/matrix), you should customize and test them depending on your needs.

**Important:** It is strongly recommended to test this notebook on a low volume of prompts first to ensure that everything works properly and that the data fits your expectations.

You can also leverage the *TestClient* and *TestClientFiltering* subclasses, which simulate API responses, to run additional tests and understand how the processing is performed.

## 1. Imports & defining functions

In [ ]:
%run -i mp_main.py
%run -i mp_utils.py

## 2. Running with an LLM API

#### How to use:

Customize the parameters of the *send_prompts_to_llm* function below based on your own context and requirements, and run the cell.

- The name of the input file containing the prompts to process should not have any extension (such as .txt)
- *batch_size, num_workers, queue_size* and *max_retries* should be customized based on your needs
- *app_name* is required if using Matrix
- *model* is required if using OpenAI or a similar API

#### Behavior illustration

Assuming *wordswap_sentence_prompts* is our input file containing prompts to process for a *generation* task.

The *send_prompts_to_llm* function will:
- Create a new folder *wordswap_sentence_prompts_batches* containing smaller text files *batch_n.txt*, representing a batch of the input file of maximum *batch_size* lines.
- For each batch, the prompts will be sent to the LLM and the responses aggregated into *batch_n_processed.txt* files
- Once the script received the responses for all batches, it will handle the responses to create *batch_n_output.txt* files. For a *filtering* task, this is when the script will automatically remove sentence pairs for which the LLM failed the filtering task.
- Finally, these files will be aggregated into the a new single file with the suffix *_final_output.txt*. In our case, it will create the file  *wordswap_sentence_prompts_final_output.txt* in the same location as the input file.

#### Need to resume after an interruption?

**In this case, do not change the batch size to avoid overwritting previous results!** Then, you can just rerun the process and it resume where it stopped.

In [ ]:
# Illustration with Matrix, to customize

await send_prompts_to_llm(
    task_type='generation', # 'generation' or 'filtering'
    input_file="home/lt-swap/data/wordswap_sentence_prompts", # Path to the file containing the prompts to process, do not use any extension for this input file
    batch_size=1_000,
    temperature=0, # Use 0.5 for agreementswap/inflectionswap sentence generation
    num_workers=20,
    queue_size=100,
    max_retries=3,
    app_name="app_name",
    client="matrix"
)


In [ ]:
# Illustration with Azure OpenAI, to customize

AZURE_API_KEY = os.getenv("AZURE_API_KEY", None)
AZURE_API_ENDPOINT = "https://your_azure_open_ai_end_point_here"

await send_prompts_to_llm(
    task_type='generation', # 'generation' or 'filtering'
    input_file="home/lt-swap/data/wordswap_sentence_prompts", # Path to the file containing the prompts to process, do not use any extension for this input file
    batch_size=1_000,
    temperature=0, # Use 0.5 for agreementswap/inflectionswap/visual sentence generation
    num_workers=20,
    queue_size=100,
    max_retries=3,
    client="azureopenai",
    model="gpt-4o",
    api_endpoint=AZURE_API_ENDPOINT,
    api_key=AZURE_API_KEY
)

In [ ]:
# Illustration with OpenAI API, to customize

await send_prompts_to_llm(
    task_type='generation', # 'generation' or 'filtering'
    input_file="home/lt-swap/data/wordswap_sentence_prompts", # Path to the file containing the prompts to process, do not use any extension for this input file
    batch_size=1_000,
    temperature=0, # Use 0.5 for agreementswap/inflectionswap sentence generation
    num_workers=20,
    queue_size=100,
    max_retries=3,
    model="gpt-5-nano",
    client="openai"
)


### 3. Testing with a "simulated" API

To better understand the behavior of the script and notebook, you can run quick tests with the "simulated" APIs, for generation and filtering tasks.

These dummy APIs simulate real-world behavior with random failures and processing delays.

- For a generation task (use client="test"), it will echo the prompt back as the response.
- For a filtering task (use client="test_filtering"), it will give a random answer between A and B.


In [ ]:
# Illustration of a test with the simulated API for a *generation* task

await send_prompts_to_llm(
    task_type='generation', # 'generation' or 'filtering'
    input_file="home/lt-swap/data/wordswap_sentence_prompts", # Path to the file containing the prompts to process, do not use any extension for this input file
    batch_size=1_000,
    temperature=0, # Use 0.5 for agreementswap/inflectionswap sentence generation
    num_workers=20,
    queue_size=100,
    max_retries=3,
    client="test"
)


In [ ]:
# Illustration of a test with the simulated API for a *filtering* task

await send_prompts_to_llm(
    task_type='filtering', # 'generation' or 'filtering'
    input_file="home/lt-swap/data/wordswap_sentence_prompts", # Path to the file containing the prompts to process, do not use any extension for this input file
    batch_size=1_000,
    temperature=0, # Use 0.5 for agreementswap/inflectionswap sentence generation
    num_workers=20,
    queue_size=100,
    max_retries=3,
    client="test_filtering"
)
